In [1]:
import pandas as pd
import numpy as np

# =========================================================
# 1. LOAD DATA
# =========================================================
df = pd.read_csv("cleaned_data.csv")

# =========================================================
# 3. BASIC HELPER METRICS
# =========================================================

# Total issues
if "total_issues" not in df.columns:
    df["total_issues"] = (
        df["number_of_open_issues"].fillna(0)
        + df["number_of_closed_issues"].fillna(0)
    )

# Total PRs
df["total_prs"] = (
    df["number_of_open_PRs"].fillna(0)
    + df["number_of_closed_PRs"].fillna(0)
)

# Issue closure rate
df["issue_closure_rate"] = np.where(
    df["total_issues"] > 0,
    df["number_of_closed_issues"].fillna(0) / df["total_issues"],
    0
)

# PR close rate
df["pr_close_rate"] = np.where(
    df["total_prs"] > 0,
    (
        df["number_of_closed_PRs"].fillna(0)
    ) / df["total_prs"],
    0
)

# =========================================================
# 4. ACTIVITY STATUS
# =========================================================
df["activity_status"] = np.where(
    (
        df["number_of_commits"].fillna(0)
        + df["number_of_closed_PRs"].fillna(0)
        + df["number_of_closed_issues"].fillna(0)
    ) == 0,
    "inactive",
    "active"
)

# =========================================================
# 5. HELPER FUNCTIONS
# =========================================================
def pct_rank(series):
    return series.rank(pct=True) * 100

def score_to_label(x):
    if x >= 60:
        return "healthy"
    else:
        return "unhealthy"

def score_to_grade(x):
    if x >= 80:
        return "A"
    elif x >= 70:
        return "B"
    elif x >= 60:
        return "C"
    elif x >= 50:
        return "D"
    else:
        return "F"

# =========================================================
# 6. COMPONENT SCORES
# =========================================================
df["commit_score"] = pct_rank(df["number_of_commits"].fillna(0))
df["closure_score"] = df["issue_closure_rate"].fillna(0) * 100
df["pr_close_score"] = df["pr_close_rate"].fillna(0) * 100
df["contributor_score"] = pct_rank(df["number_of_contributors"].fillna(0))

# =========================================================
# 8. FINAL HEALTH SCORE
# =========================================================
df["health_score"] = (
    0.35* df["commit_score"]
    + 0.3 * df["closure_score"]
    + 0.2 * df["pr_close_score"]
    + 0.15 * df["contributor_score"]
).clip(0, 100)

weight_schemes = {
    "equal": (0.25, 0.25, 0.25, 0.25),
    "commit_heavy": (0.50, 0.20, 0.15, 0.15),
    "closure_heavy": (0.25, 0.40, 0.20, 0.15),
    "contributor_heavy": (0.25, 0.25, 0.15, 0.35)
}

for scheme_name, (w_commit, w_closure, w_pr, w_contributor) in weight_schemes.items():
    score_col = f"health_score_{scheme_name}"
    label_col = f"health_label_{scheme_name}"
    
    df[score_col] = (
        w_commit * df["commit_score"]
        + w_closure * df["closure_score"]
        + w_pr * df["pr_close_score"]
        + w_contributor * df["contributor_score"]
    ).clip(0, 100)
    
    df[label_col] = df[score_col].apply(score_to_label)

# =========================================================
# 9. HEALTH GRADE + LABEL
# =========================================================
df["health_grade"] = df["health_score"].apply(score_to_grade)
df["health_label"] = df["health_score"].apply(score_to_label)

# =========================================================
# 10. OUTPUT
# =========================================================
output_cols = [
    "repo",
    "year",
    "month",
    "number_of_commits",
    "number_of_contributors",
    "number_of_open_PRs",
    "number_of_closed_PRs",
    "number_of_merged_PRs",
    "number_of_open_issues",
    "number_of_closed_issues",
    "days_since_last_commit",
    "issue_closure_rate",
    "pr_close_rate",
    "commit_score",
    "closure_score",
    "pr_close_score",
    "contributor_score",
    "activity_status",
    "health_score",
    "health_grade",
    "health_label",
    "health_score_equal",
    "health_label_equal",
    "health_score_commit_heavy",
    "health_label_commit_heavy",
    "health_score_closure_heavy",
    "health_label_closure_heavy",
    "health_score_contributor_heavy",
    "health_label_contributor_heavy"
]

result = df[output_cols].copy()

# =========================================================
# 11. SAVE
# =========================================================
result.to_csv("finalize.csv", index=False)

In [2]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# =========================================================
# 1. LOAD DATA
# =========================================================
df = pd.read_csv("cleaned_data.csv")

# =========================================================
# 2. BASIC METRICS
# =========================================================
df["number_of_commits"] = df["number_of_commits"].fillna(0)
df["number_of_closed_PRs"] = df["number_of_closed_PRs"].fillna(0)
df["number_of_merged_PRs"] = df["number_of_merged_PRs"].fillna(0)
df["number_of_open_PRs"] = df["number_of_open_PRs"].fillna(0)
df["number_of_closed_issues"] = df["number_of_closed_issues"].fillna(0)
df["number_of_open_issues"] = df["number_of_open_issues"].fillna(0)
df["number_of_contributors"] = df["number_of_contributors"].fillna(0)

df["total_issues"] = df["number_of_open_issues"] + df["number_of_closed_issues"]
df["total_prs"] = df["number_of_open_PRs"] + df["number_of_closed_PRs"]

df["issue_closure_rate"] = np.where(
    df["total_issues"] > 0,
    df["number_of_closed_issues"] / df["total_issues"],
    0
)

df["pr_close_rate"] = np.where(
    df["total_prs"] > 0,
    (df["number_of_closed_PRs"])/ df["total_prs"],
    0
)

# =========================================================
# 3. PERCENTILE SCORES
# =========================================================
def pct_rank(series):
    return series.rank(pct=True) * 100

df["commit_score"] = pct_rank(df["number_of_commits"])
df["closure_score"] = df["issue_closure_rate"] * 100
df["pr_close_score"] = df["pr_close_rate"] * 100
df["contributor_score"] = pct_rank(df["number_of_contributors"])

features = ["commit_score", "closure_score", "pr_close_score", "contributor_score"]

# =========================================================
# 4. BUILD FUTURE TARGET
# =========================================================
df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
df = df.sort_values(["repo", "date"]).reset_index(drop=True)

# current activity in month t
df["total_activity"] = (
    df["number_of_commits"]
    + df["number_of_closed_PRs"]
    + df["number_of_closed_issues"]
)

g = df.groupby("repo")["total_activity"]
f1 = g.shift(-1)
f2 = g.shift(-2)
f3 = g.shift(-3)

# future 3-month activity
df["future_activity_3m"] = np.where(
    f1.notna() & f2.notna() & f3.notna(),
    f1 + f2 + f3,
    np.nan
)

# remove rows without future target
df = df.dropna(subset=["future_activity_3m"]).copy()

# log transform to reduce skew
df["future_activity_3m_log"] = np.log1p(df["future_activity_3m"])

# =========================================================
# 5. WEIGHT OPTIMIZATION
# =========================================================
def fit_optimized_weights(X_train, y_train):
    """
    Find non-negative weights summing to 1 that minimize RMSE.
    """
    n_features = X_train.shape[1]

    def objective(w):
        pred = X_train @ w
        return np.sqrt(mean_squared_error(y_train, pred))

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
    bounds = [(0, 1)] * n_features
    w0 = np.repeat(1 / n_features, n_features)

    result = minimize(
        objective,
        x0=w0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints
    )

    if not result.success:
        raise ValueError(f"Optimization failed: {result.message}")

    return result.x


def evaluate_predictions(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

# =========================================================
# 6. BASELINE WEIGHT SCHEMES
# =========================================================
baseline_weights = {
    "current_manual": np.array([0.35, 0.30, 0.20, 0.15]),
    "equal": np.array([0.25, 0.25, 0.25, 0.25]),
    "commit_heavy": np.array([0.50, 0.20, 0.15, 0.15]),
    "closure_heavy": np.array([0.25, 0.40, 0.20, 0.15]),
    "contributor_heavy": np.array([0.25, 0.25, 0.15, 0.35]),
}

# =========================================================
# 7. EVALUATE ONE SPLIT
# =========================================================
def run_one_split(train_df, test_df, split_name):
    results = []
    weight_rows = []

    X_train = train_df[features].values
    X_test = test_df[features].values

    # Scale y to 0-100 based only on train set
    scaler = MinMaxScaler(feature_range=(0, 100))
    y_train = scaler.fit_transform(train_df[["future_activity_3m_log"]]).ravel()
    y_test = scaler.transform(test_df[["future_activity_3m_log"]]).ravel()

    # ---------- optimized weights ----------
    opt_w = fit_optimized_weights(X_train, y_train)
    y_pred_opt = X_test @ opt_w
    metrics_opt = evaluate_predictions(y_test, y_pred_opt)

    results.append({
        "split": split_name,
        "model": "optimized_weights",
        **metrics_opt
    })

    weight_rows.append({
        "split": split_name,
        "model": "optimized_weights",
        "commit_weight": opt_w[0],
        "closure_weight": opt_w[1],
        "pr_weight": opt_w[2],
        "contributor_weight": opt_w[3]
    })

    # ---------- manual baseline schemes ----------
    for name, w in baseline_weights.items():
        y_pred = X_test @ w
        metrics = evaluate_predictions(y_test, y_pred)

        results.append({
            "split": split_name,
            "model": name,
            **metrics
        })

        weight_rows.append({
            "split": split_name,
            "model": name,
            "commit_weight": w[0],
            "closure_weight": w[1],
            "pr_weight": w[2],
            "contributor_weight": w[3]
        })

    # ---------- linear regression (supporting evidence) ----------
    lr = LinearRegression(positive=True)
    lr.fit(X_train, y_train)
    y_pred_lr = lr.predict(X_test)
    metrics_lr = evaluate_predictions(y_test, y_pred_lr)

    coef = np.maximum(lr.coef_, 0)
    if coef.sum() > 0:
        lr_w = coef / coef.sum()
    else:
        lr_w = np.repeat(0.25, len(features))

    results.append({
        "split": split_name,
        "model": "linear_regression",
        **metrics_lr
    })

    weight_rows.append({
        "split": split_name,
        "model": "linear_regression",
        "commit_weight": lr_w[0],
        "closure_weight": lr_w[1],
        "pr_weight": lr_w[2],
        "contributor_weight": lr_w[3]
    })

    # ---------- elastic net (regularized supporting evidence) ----------
    # Standardize X only for ElasticNet so coefficients are more comparable.
    x_scaler = StandardScaler()
    X_train_en = x_scaler.fit_transform(X_train)
    X_test_en = x_scaler.transform(X_test)

    en = ElasticNet(
        alpha=0.10,
        l1_ratio=0.50,
        positive=True,
        max_iter=20000,
        random_state=42
    )
    en.fit(X_train_en, y_train)
    y_pred_en = en.predict(X_test_en)
    metrics_en = evaluate_predictions(y_test, y_pred_en)

    en_coef = np.maximum(en.coef_, 0)
    if en_coef.sum() > 0:
        en_w = en_coef / en_coef.sum()
    else:
        en_w = np.repeat(0.25, len(features))

    results.append({
        "split": split_name,
        "model": "elastic_net",
        **metrics_en
    })

    weight_rows.append({
        "split": split_name,
        "model": "elastic_net",
        "commit_weight": en_w[0],
        "closure_weight": en_w[1],
        "pr_weight": en_w[2],
        "contributor_weight": en_w[3]
    })

    # ---------- random forest (importance only, not final weights) ----------
    rf = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=5,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    metrics_rf = evaluate_predictions(y_test, y_pred_rf)

    imp = rf.feature_importances_
    rf_w = imp / imp.sum()

    results.append({
        "split": split_name,
        "model": "random_forest",
        **metrics_rf
    })

    weight_rows.append({
        "split": split_name,
        "model": "random_forest_importance",
        "commit_weight": rf_w[0],
        "closure_weight": rf_w[1],
        "pr_weight": rf_w[2],
        "contributor_weight": rf_w[3]
    })

    # ---------- gradient boosting (nonlinear supporting evidence) ----------
    gbr = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        random_state=42
    )
    gbr.fit(X_train, y_train)
    y_pred_gbr = gbr.predict(X_test)
    metrics_gbr = evaluate_predictions(y_test, y_pred_gbr)

    gbr_imp = gbr.feature_importances_
    if gbr_imp.sum() > 0:
        gbr_w = gbr_imp / gbr_imp.sum()
    else:
        gbr_w = np.repeat(0.25, len(features))

    results.append({
        "split": split_name,
        "model": "gradient_boosting",
        **metrics_gbr
    })

    weight_rows.append({
        "split": split_name,
        "model": "gradient_boosting_importance",
        "commit_weight": gbr_w[0],
        "closure_weight": gbr_w[1],
        "pr_weight": gbr_w[2],
        "contributor_weight": gbr_w[3]
    })

    return pd.DataFrame(results), pd.DataFrame(weight_rows)

# =========================================================
# 8. TEMPORAL SPLITS
# =========================================================
all_results = []
all_weights = []

# Rolling split
for test_year in [2021, 2022, 2023, 2024, 2025]:
    train_df = df[df["year"] < test_year].copy()
    test_df = df[df["year"] == test_year].copy()

    if len(train_df) == 0 or len(test_df) == 0:
        continue

    r, w = run_one_split(train_df, test_df, f"rolling_test_{test_year}")
    all_results.append(r)
    all_weights.append(w)

# Holdout split
train_df = df[df["year"].between(2020, 2023)].copy()
test_df = df[df["year"].between(2024, 2025)].copy()

if len(train_df) > 0 and len(test_df) > 0:
    r, w = run_one_split(train_df, test_df, "holdout_2020_2023_vs_2024_2025")
    all_results.append(r)
    all_weights.append(w)

results_df = pd.concat(all_results, ignore_index=True)
weights_df = pd.concat(all_weights, ignore_index=True)

# =========================================================
# 9. SUMMARY
# =========================================================
summary = (
    results_df
    .groupby("model")[["RMSE", "MAE", "R2"]]
    .mean()
    .sort_values(["RMSE", "MAE", "R2"], ascending=[True, True, False])
    .reset_index()
)

avg_weights = (
    weights_df
    .groupby("model")[["commit_weight", "closure_weight", "pr_weight", "contributor_weight"]]
    .mean()
    .reset_index()
)

print("\n=== Average performance across temporal splits ===")
print(summary)

print("\n=== Average learned / compared weights ===")
print(avg_weights)

results_df.to_csv("weight_model_results.csv", index=False)
weights_df.to_csv("weight_values_by_split.csv", index=False)
summary.to_csv("weight_model_summary.csv", index=False)
avg_weights.to_csv("average_weights_summary.csv", index=False)



=== Average performance across temporal splits ===
               model       RMSE        MAE        R2
0  gradient_boosting  12.051635   8.845862  0.724660
1      random_forest  12.145713   8.931603  0.721409
2  linear_regression  13.086059   9.627323  0.677954
3        elastic_net  13.148925   9.708011  0.675067
4  optimized_weights  17.010956  12.429887  0.445494
5       commit_heavy  18.229913  14.668033  0.364004
6  contributor_heavy  19.216556  15.609532  0.289351
7     current_manual  20.159967  16.867708  0.223443
8              equal  20.184978  16.791309  0.219281
9      closure_heavy  22.011499  18.628122  0.075623

=== Average learned / compared weights ===
                          model  commit_weight  closure_weight  pr_weight  \
0                 closure_heavy       0.250000        0.400000   0.200000   
1                  commit_heavy       0.500000        0.200000   0.150000   
2             contributor_heavy       0.250000        0.250000   0.150000   
3            

In [3]:
#FIND THRESHOLD
import numpy as np
import pandas as pd

df_threshold = df.copy()

#final health definition
df_threshold["health_score_final"] = (
    0.60 * df_threshold["commit_score"] +
    0.10 * df_threshold["closure_score"] +
    0.15 * df_threshold["pr_close_score"] +
    0.15 * df_threshold["contributor_score"]
).clip(0, 100)

# thử threshold
threshold = 60

df_threshold["health_label_final"] = np.where(
    df_threshold["health_score_final"] >= threshold,
    "healthy",
    "unhealthy"
)

summary = pd.DataFrame({
    "count": df_threshold["health_label_final"].value_counts(),
    "percent": (df_threshold["health_label_final"].value_counts(normalize=True) * 100).round(2)
})

print("Threshold =", threshold)
print(summary)

Threshold = 60
                    count  percent
health_label_final                
unhealthy           12622    64.52
healthy              6942    35.48


In [ ]:
#TEST ACTIVITY STATUS (RQ2)
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

# =========================================================
# 1. LOAD DATA
# =========================================================
df = pd.read_csv("cleaned_data.csv")

# =========================================================
# 2. BASIC CLEANING
# =========================================================
df["number_of_commits"] = df["number_of_commits"].fillna(0)
df["number_of_closed_PRs"] = df["number_of_closed_PRs"].fillna(0)
df["number_of_merged_PRs"] = df["number_of_merged_PRs"].fillna(0)
df["number_of_open_PRs"] = df["number_of_open_PRs"].fillna(0)
df["number_of_closed_issues"] = df["number_of_closed_issues"].fillna(0)
df["number_of_open_issues"] = df["number_of_open_issues"].fillna(0)
df["number_of_contributors"] = df["number_of_contributors"].fillna(0)

# =========================================================
# 3. BASIC HELPER METRICS
# =========================================================
df["total_issues"] = df["number_of_open_issues"] + df["number_of_closed_issues"]
df["total_prs"] = df["number_of_open_PRs"] + df["number_of_closed_PRs"]

df["issue_closure_rate"] = np.where(
    df["total_issues"] > 0,
    df["number_of_closed_issues"] / df["total_issues"],
    0
)

df["pr_close_rate"] = np.where(
    df["total_prs"] > 0,
    (df["number_of_closed_PRs"]) / df["total_prs"],
    0
)

# =========================================================
# 4. SCORE FEATURES
# =========================================================
def pct_rank(series):
    return series.rank(pct=True) * 100

df["commit_score"] = pct_rank(df["number_of_commits"])
df["closure_score"] = df["issue_closure_rate"] * 100
df["pr_close_score"] = df["pr_close_rate"] * 100
df["contributor_score"] = pct_rank(df["number_of_contributors"])

features = ["commit_score", "closure_score", "pr_close_score", "contributor_score"]

# =========================================================
# 5. BUILD TIME ORDER + CURRENT ACTIVITY
# =========================================================
df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
df = df.sort_values(["repo", "date"]).reset_index(drop=True)

# current activity in month t
df["total_activity"] = (
    df["number_of_commits"]
    + df["number_of_closed_PRs"]
    + df["number_of_closed_issues"]
)

# keep original/raw current activity status for reference only
df["activity_status_current_raw"] = np.where(df["total_activity"] > 0, 1, 0)

# =========================================================
# 6. BUILD FUTURE 3-MONTH ACTIVITY
# =========================================================
g = df.groupby("repo")["total_activity"]

f1 = g.shift(-1)
f2 = g.shift(-2)
f3 = g.shift(-3)

df["activity_m1"] = f1
df["activity_m2"] = f2
df["activity_m3"] = f3

df["future_activity_3m"] = np.where(
    f1.notna() & f2.notna() & f3.notna(),
    f1 + f2 + f3,
    np.nan
)

# keep rows where future target can be computed
df = df.dropna(subset=["future_activity_3m"]).copy()

# =========================================================
# 7. BUILD NEW RQ2 LABEL
#    stricter future activity status:
#    - total future activity >= 70
#    - active in at least 2 of next 3 months
# =========================================================
df["active_months_next3"] = (
    (df["activity_m1"] > 0).astype(int)
    + (df["activity_m2"] > 0).astype(int)
    + (df["activity_m3"] > 0).astype(int)
)

df["future_activity_status_rq2"] = np.where(
    (df["future_activity_3m"] >= 70) & (df["active_months_next3"] >= 2),
    1,   # active
    0    # inactive
)

# inspect class balance
overall_class_summary = pd.DataFrame({
    "count": df["future_activity_status_rq2"].value_counts().sort_index(),
    "percent": (df["future_activity_status_rq2"].value_counts(normalize=True).sort_index() * 100).round(2)
})
overall_class_summary.index = ["inactive", "active"]

print("\n=== Overall class distribution for RQ2 label ===")
print(overall_class_summary)

# =========================================================
# 8. EVALUATION FUNCTION
# =========================================================
def evaluate_cls(y_true, y_pred, y_prob=None):
    out = {
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }

    if y_prob is not None and len(np.unique(y_true)) > 1:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    else:
        out["roc_auc"] = np.nan

    return out

# =========================================================
# 9. RUN ONE TEMPORAL SPLIT FOR RQ2
# =========================================================
def run_one_split_rq2(train_df, test_df, split_name):
    results = []

    # skip degenerate splits
    if train_df["future_activity_status_rq2"].nunique() < 2:
        return pd.DataFrame()
    if test_df["future_activity_status_rq2"].nunique() < 2:
        return pd.DataFrame()

    X_train = train_df[features].values
    X_test = test_df[features].values

    y_train = train_df["future_activity_status_rq2"].astype(int).values
    y_test = test_df["future_activity_status_rq2"].astype(int).values

    train_active_rate = y_train.mean()
    test_active_rate = y_test.mean()

    # -----------------------------------------------------
    # 9.1 Dummy baseline
    # -----------------------------------------------------
    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(X_train, y_train)

    pred_dummy = dummy.predict(X_test)

    res = evaluate_cls(y_test, pred_dummy, None)
    results.append({
        "split": split_name,
        "model": "dummy_most_frequent",
        "train_active_rate": train_active_rate,
        "test_active_rate": test_active_rate,
        **res
    })

    # -----------------------------------------------------
    # 9.2 Logistic Regression
    # -----------------------------------------------------
    scaler = StandardScaler()
    X_train_lr = scaler.fit_transform(X_train)
    X_test_lr = scaler.transform(X_test)

    lr = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    )
    lr.fit(X_train_lr, y_train)

    pred_lr = lr.predict(X_test_lr)
    prob_lr = lr.predict_proba(X_test_lr)[:, 1]

    res = evaluate_cls(y_test, pred_lr, prob_lr)
    results.append({
        "split": split_name,
        "model": "logistic_regression",
        "train_active_rate": train_active_rate,
        "test_active_rate": test_active_rate,
        **res
    })

    # -----------------------------------------------------
    # 9.3 Random Forest
    # -----------------------------------------------------
    rf = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    pred_rf = rf.predict(X_test)
    prob_rf = rf.predict_proba(X_test)[:, 1]

    res = evaluate_cls(y_test, pred_rf, prob_rf)
    results.append({
        "split": split_name,
        "model": "random_forest_classifier",
        "train_active_rate": train_active_rate,
        "test_active_rate": test_active_rate,
        **res
    })

    # -----------------------------------------------------
    # 9.4 Gradient Boosting
    # -----------------------------------------------------
    gb = GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
    gb.fit(X_train, y_train)

    pred_gb = gb.predict(X_test)
    prob_gb = gb.predict_proba(X_test)[:, 1]

    res = evaluate_cls(y_test, pred_gb, prob_gb)
    results.append({
        "split": split_name,
        "model": "gradient_boosting_classifier",
        "train_active_rate": train_active_rate,
        "test_active_rate": test_active_rate,
        **res
    })

    return pd.DataFrame(results)

# =========================================================
# 10. TEMPORAL SPLITS
#    same style as your current regression script
# =========================================================
all_results = []

# rolling split
for test_year in [2021, 2022, 2023, 2024, 2025]:
    train_df = df[df["year"] < test_year].copy()
    test_df = df[df["year"] == test_year].copy()

    if len(train_df) == 0 or len(test_df) == 0:
        continue

    r = run_one_split_rq2(train_df, test_df, f"rolling_test_{test_year}")
    if not r.empty:
        all_results.append(r)

# holdout split
train_df = df[df["year"].between(2020, 2023)].copy()
test_df = df[df["year"].between(2024, 2025)].copy()

if len(train_df) > 0 and len(test_df) > 0:
    r = run_one_split_rq2(train_df, test_df, "holdout_2020_2023_vs_2024_2025")
    if not r.empty:
        all_results.append(r)

if len(all_results) == 0:
    raise ValueError("No valid temporal split produced both classes for RQ2. Check the label definition.")

rq2_results = pd.concat(all_results, ignore_index=True)

# =========================================================
# 11. SUMMARY
# =========================================================
rq2_summary = (
    rq2_results
    .groupby("model")[["balanced_accuracy", "f1", "precision", "recall", "roc_auc"]]
    .mean()
    .sort_values(["balanced_accuracy", "f1"], ascending=False)
    .reset_index()
)

split_class_balance = (
    rq2_results[["split", "train_active_rate", "test_active_rate"]]
    .drop_duplicates()
    .sort_values("split")
    .reset_index(drop=True)
)

print("\n=== RQ2 classification summary ===")
print(rq2_summary)

print("\n=== Split class balance ===")
print(split_class_balance)

# =========================================================
# 12. SAVE OUTPUTS
# =========================================================
df.to_csv("rq2_labeled_dataset.csv", index=False)
rq2_results.to_csv("rq2_future_activity_status_results.csv", index=False)
rq2_summary.to_csv("rq2_future_activity_status_summary.csv", index=False)
split_class_balance.to_csv("rq2_split_class_balance.csv", index=False)

print("\nSaved files:")
print("- rq2_labeled_dataset.csv")
print("- rq2_future_activity_status_results.csv")
print("- rq2_future_activity_status_summary.csv")
print("- rq2_split_class_balance.csv")


=== Overall class distribution for RQ2 label ===
          count  percent
inactive   5963    30.48
active    13601    69.52

=== RQ2 classification summary ===
                          model  balanced_accuracy        f1  precision  \
0      random_forest_classifier           0.857006  0.892312   0.931971   
1           logistic_regression           0.846630  0.878708   0.931811   
2  gradient_boosting_classifier           0.831362  0.898760   0.897137   
3           dummy_most_frequent           0.500000  0.819080   0.694084   

     recall   roc_auc  
0  0.856006  0.927068  
1  0.831598  0.920915  
2  0.900434  0.924391  
3  1.000000       NaN  

=== Split class balance ===
                            split  train_active_rate  test_active_rate
0  holdout_2020_2023_vs_2024_2025           0.717230          0.674584
1               rolling_test_2021           0.693452          0.690948
2               rolling_test_2022           0.691611          0.724424
3               rolling_test_2

In [5]:
#TEST THRESHOLD AND PERSISTENT
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# =========================================================
# 1. LOAD + PREP
# =========================================================
df = pd.read_csv("cleaned_data.csv")

activity_cols = [
    "number_of_commits",
    "number_of_closed_PRs",
    "number_of_merged_PRs",
    "number_of_open_PRs",
    "number_of_closed_issues",
    "number_of_open_issues",
    "number_of_contributors"
]

for col in activity_cols:
    df[col] = df[col].fillna(0)

df["total_issues"] = df["number_of_open_issues"] + df["number_of_closed_issues"]
df["total_prs"] = df["number_of_open_PRs"] + df["number_of_closed_PRs"]

df["issue_closure_rate"] = np.where(
    df["total_issues"] > 0,
    df["number_of_closed_issues"] / df["total_issues"],
    0
)

df["pr_close_rate"] = np.where(
    df["total_prs"] > 0,
    (df["number_of_closed_PRs"]) / df["total_prs"],
    0
)

def pct_rank(series):
    return series.rank(pct=True) * 100

df["commit_score"] = pct_rank(df["number_of_commits"])
df["closure_score"] = df["issue_closure_rate"] * 100
df["pr_close_score"] = df["pr_close_rate"] * 100
df["contributor_score"] = pct_rank(df["number_of_contributors"])

features = ["commit_score", "closure_score", "pr_close_score", "contributor_score"]

df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
df = df.sort_values(["repo", "date"]).reset_index(drop=True)

# activity formula bạn đang dùng
df["total_activity"] = (
    df["number_of_commits"]
    + df["number_of_closed_PRs"]
    + df["number_of_closed_issues"]
)

g = df.groupby("repo")["total_activity"]

df["activity_m1"] = g.shift(-1)
df["activity_m2"] = g.shift(-2)
df["activity_m3"] = g.shift(-3)

df["future_activity_3m"] = np.where(
    df["activity_m1"].notna() & df["activity_m2"].notna() & df["activity_m3"].notna(),
    df["activity_m1"] + df["activity_m2"] + df["activity_m3"],
    np.nan
)

df["active_months_next3"] = (
    (df["activity_m1"] > 0).astype(int)
    + (df["activity_m2"] > 0).astype(int)
    + (df["activity_m3"] > 0).astype(int)
)

df = df.dropna(subset=["future_activity_3m"]).copy()

# =========================================================
# 2. METRIC FUNCTION
# =========================================================
def evaluate_cls(y_true, y_pred, y_prob=None):
    out = {
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }
    if y_prob is not None and len(np.unique(y_true)) > 1:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    else:
        out["roc_auc"] = np.nan
    return out

# =========================================================
# 3. RUN ONE SPLIT
# =========================================================
def run_one_split(train_df, test_df, split_name, target_col):
    results = []

    if train_df[target_col].nunique() < 2 or test_df[target_col].nunique() < 2:
        return pd.DataFrame()

    X_train = train_df[features].values
    X_test = test_df[features].values
    y_train = train_df[target_col].astype(int).values
    y_test = test_df[target_col].astype(int).values

    train_active_rate = y_train.mean()
    test_active_rate = y_test.mean()

    # Dummy
    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(X_train, y_train)
    pred_dummy = dummy.predict(X_test)
    res = evaluate_cls(y_test, pred_dummy, None)
    results.append({
        "split": split_name,
        "model": "dummy",
        "train_active_rate": train_active_rate,
        "test_active_rate": test_active_rate,
        **res
    })

    # Logistic
    scaler = StandardScaler()
    X_train_lr = scaler.fit_transform(X_train)
    X_test_lr = scaler.transform(X_test)

    lr = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)
    lr.fit(X_train_lr, y_train)
    pred_lr = lr.predict(X_test_lr)
    prob_lr = lr.predict_proba(X_test_lr)[:, 1]
    res = evaluate_cls(y_test, pred_lr, prob_lr)
    results.append({
        "split": split_name,
        "model": "logistic_regression",
        "train_active_rate": train_active_rate,
        "test_active_rate": test_active_rate,
        **res
    })

    # RF
    rf = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    pred_rf = rf.predict(X_test)
    prob_rf = rf.predict_proba(X_test)[:, 1]
    res = evaluate_cls(y_test, pred_rf, prob_rf)
    results.append({
        "split": split_name,
        "model": "random_forest",
        "train_active_rate": train_active_rate,
        "test_active_rate": test_active_rate,
        **res
    })

    # GB
    gb = GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
    gb.fit(X_train, y_train)
    pred_gb = gb.predict(X_test)
    prob_gb = gb.predict_proba(X_test)[:, 1]
    res = evaluate_cls(y_test, pred_gb, prob_gb)
    results.append({
        "split": split_name,
        "model": "gradient_boosting",
        "train_active_rate": train_active_rate,
        "test_active_rate": test_active_rate,
        **res
    })

    return pd.DataFrame(results)

# =========================================================
# 4. EVALUATE ONE LABEL DEFINITION
# =========================================================
def evaluate_label_definition(df_in, threshold, persistence):
    df_eval = df_in.copy()

    target_col = f"label_t{threshold}_p{persistence}"
    df_eval[target_col] = np.where(
        (df_eval["future_activity_3m"] >= threshold) &
        (df_eval["active_months_next3"] >= persistence),
        1, 0
    )

    class_rate = df_eval[target_col].mean()

    all_results = []

    # rolling splits
    for test_year in [2021, 2022, 2023, 2024, 2025]:
        train_df = df_eval[df_eval["year"] < test_year].copy()
        test_df = df_eval[df_eval["year"] == test_year].copy()

        if len(train_df) == 0 or len(test_df) == 0:
            continue

        r = run_one_split(train_df, test_df, f"rolling_test_{test_year}", target_col)
        if not r.empty:
            all_results.append(r)

    # holdout
    train_df = df_eval[df_eval["year"].between(2020, 2023)].copy()
    test_df = df_eval[df_eval["year"].between(2024, 2025)].copy()

    if len(train_df) > 0 and len(test_df) > 0:
        r = run_one_split(train_df, test_df, "holdout_2020_2023_vs_2024_2025", target_col)
        if not r.empty:
            all_results.append(r)

    if len(all_results) == 0:
        return None

    results_df = pd.concat(all_results, ignore_index=True)

    summary_df = (
        results_df
        .groupby("model")[["balanced_accuracy", "f1", "precision", "recall", "roc_auc"]]
        .mean()
        .reset_index()
    )

    summary_df["threshold"] = threshold
    summary_df["persistence"] = persistence
    summary_df["overall_active_rate"] = class_rate
    summary_df["num_rows"] = len(df_eval)

    return summary_df, results_df

# =========================================================
# 5. GRID SEARCH OVER THRESHOLD + PERSISTENCE
# =========================================================
thresholds = [30, 50, 70, 100]
persistences = [1, 2, 3]

all_summaries = []
all_detail_results = []

for t in thresholds:
    for p in persistences:
        out = evaluate_label_definition(df, t, p)

        if out is None:
            print(f"Skipping threshold={t}, persistence={p} (degenerate splits)")
            continue

        summary_df, detail_df = out
        all_summaries.append(summary_df)
        all_detail_results.append(detail_df)

final_summary = pd.concat(all_summaries, ignore_index=True)
final_details = pd.concat(all_detail_results, ignore_index=True)

# =========================================================
# 6. FILTER TO REAL MODELS ONLY (optional)
# =========================================================
real_model_summary = final_summary[final_summary["model"] != "dummy"].copy()

# sort by balanced accuracy, then roc_auc
real_model_summary = real_model_summary.sort_values(
    ["balanced_accuracy", "roc_auc", "f1"],
    ascending=False
)

print("\n=== Threshold/Persistence Comparison Summary ===")
print(real_model_summary)

final_summary.to_csv("threshold_persistence_summary.csv", index=False)
final_details.to_csv("threshold_persistence_details.csv", index=False)

print("\nSaved:")
print("- threshold_persistence_summary.csv")
print("- threshold_persistence_details.csv")


=== Threshold/Persistence Comparison Summary ===
                  model  balanced_accuracy        f1  precision    recall  \
11        random_forest           0.862477  0.914778   0.952483  0.879982   
23        random_forest           0.862104  0.907112   0.940068  0.876412   
7         random_forest           0.861784  0.915496   0.955173  0.879031   
3         random_forest           0.861079  0.915221   0.955460  0.878291   
19        random_forest           0.859647  0.904910   0.941686  0.870944   
15        random_forest           0.859526  0.905033   0.941669  0.871201   
35        random_forest           0.857234  0.893561   0.929372  0.860442   
31        random_forest           0.855393  0.891750   0.930665  0.856057   
27        random_forest           0.855359  0.891696   0.930770  0.855873   
47        random_forest           0.854011  0.879272   0.918264  0.843638   
43        random_forest           0.853591  0.878677   0.919430  0.841586   
2   logistic_regression   